# 📞 SF 311 Cases - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests **San Francisco 311 cases** into Elasticsearch: every request San Franciscans file about street cleaning, graffiti, potholes, noise, blocked sidewalks, broken streetlights, and more. Many cases carry a **photo** taken by the reporter.

**What this notebook does:**
1. Fetches recent cases straight from [DataSF](https://data.sf.gov/d/vw6y-z8j6) - no API key required
2. Normalises each case (dates, neighborhood, `geo_point`, hours to close, photo URL)
3. Bulk-indexes everything into `sf_311_cases`
4. **(Optional)** asks a vision-capable **Mistral** model to describe a case photo

---
> **Data source:** 311 Cases, dataset `vw6y-z8j6`, served over the public Socrata JSON API. The full set has millions of rows going back to 2008, so we grab a recent window.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from datetime import datetime, timedelta
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch recent 311 cases
We grab the most recent `SAMPLE_SIZE` cases from the last `DAYS` days. **Raise them** for more history, or add to the `$where` clause to focus on one `service_name` (e.g. `Encampment`, `Noise`, `Tree Maintenance`) or neighborhood.

In [ ]:
SAMPLE_SIZE = 50000   # rows to fetch; raise for more coverage, lower for a quicker run
DAYS        = 60

def fetch_socrata(dataset_id, limit, page_size=10000, **params):
    """Page through a DataSF dataset with $limit/$offset until `limit` rows or the end."""
    url = f'https://data.sf.gov/resource/{dataset_id}.json'
    rows = []
    while len(rows) < limit:
        page = {**params, '$limit': min(page_size, limit - len(rows)), '$offset': len(rows)}
        r = requests.get(url, params=page, timeout=120)
        r.raise_for_status()
        batch = r.json()
        rows.extend(batch)
        print(f'  fetched {len(rows)} rows...', end='\r', flush=True)
        if len(batch) < page['$limit']:
            break
    return rows

since = (datetime.now() - timedelta(days=DAYS)).strftime('%Y-%m-%dT00:00:00')
rows = fetch_socrata('vw6y-z8j6', SAMPLE_SIZE,
                     **{'$where': f"requested_datetime > '{since}'", '$order': 'requested_datetime DESC'})

print(f'\n✅ Fetched {len(rows)} cases since {since[:10]}')
print()
print('Sample row fields:')
for k, v in list(rows[0].items())[:14]:
    print(f'  {k:32s} {str(v)[:60]}')

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'sf_311_cases'

mapping = {'mappings': {'properties': {
    'service_request_id': {'type': 'keyword'},
    'requested_at':       {'type': 'date'},
    'closed_at':          {'type': 'date'},
    'hours_to_close':     {'type': 'float'},
    'status':             {'type': 'keyword'},   # Open / Closed
    'status_notes':       {'type': 'text'},      # what the city did about it
    'agency':             {'type': 'keyword'},
    'service_name':       {'type': 'keyword'},   # Street and Sidewalk Cleaning, Graffiti, Noise, ...
    'service_subtype':    {'type': 'keyword'},
    'service_details':    {'type': 'text'},
    'address':            {'type': 'text'},
    'street':             {'type': 'keyword'},
    'neighborhood':       {'type': 'keyword'},   # one of SF's 41 analysis neighborhoods
    'supervisor_district':{'type': 'keyword'},
    'police_district':    {'type': 'keyword'},
    'source':             {'type': 'keyword'},   # Mobile, Phone, Web, ...
    'photo_url':          {'type': 'keyword'},
    'location':           {'type': 'geo_point'},
}}}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')
es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest
We parse the dates, compute `hours_to_close`, tidy the district number, and keep only photo links that are publicly viewable (the `cloudinary` ones; the others need a city login).

In [ ]:
def geo_point(lat, lon):
    """Build a geo_point from lat/lon strings, skipping blanks and 0,0."""
    try:
        lat_f, lon_f = float(lat), float(lon)
    except (TypeError, ValueError):
        return None
    if lat_f == 0.0 or lon_f == 0.0:
        return None
    return {'lat': lat_f, 'lon': lon_f}

def parse_dt(v):
    try:
        return datetime.fromisoformat(v)
    except (TypeError, ValueError):
        return None

def enrich(r):
    opened, closed = parse_dt(r.get('requested_datetime')), parse_dt(r.get('closed_date'))
    photo = (r.get('media_url') or {}).get('url', '')
    doc = {
        'service_request_id': r.get('service_request_id'),
        'requested_at':       r.get('requested_datetime'),
        'closed_at':          r.get('closed_date'),
        'hours_to_close':     round((closed - opened).total_seconds() / 3600, 2) if opened and closed else None,
        'status':             r.get('status_description'),
        'status_notes':       r.get('status_notes'),
        'agency':             r.get('agency_responsible'),
        'service_name':       r.get('service_name'),
        'service_subtype':    r.get('service_subtype'),
        'service_details':    r.get('service_details'),
        'address':            r.get('address'),
        'street':             r.get('street'),
        'neighborhood':       r.get('analysis_neighborhood'),
        'supervisor_district':(r.get('supervisor_district') or '').replace('.0', ''),
        'police_district':    r.get('police_district'),
        'source':             r.get('source'),
        'photo_url':          photo if 'cloudinary' in photo else None,
        'location':           geo_point(r.get('lat'), r.get('long')),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in rows]
success, errors = helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} documents into {INDEX} ({len(errors)} errors)')
print(f"   Cases with a public photo: {sum(1 for d in docs if 'photo_url' in d)}")

## 5. Verify - what is SF asking for?

In [ ]:
resp = es.search(index=INDEX, body={'size': 0, 'track_total_hits': True, 'aggs': {
    'by_service':      {'terms': {'field': 'service_name', 'size': 10}},
    'by_neighborhood': {'terms': {'field': 'neighborhood', 'size': 10}},
}})
print(f"Total cases: {resp['hits']['total']['value']}\n")
print('Top request types:')
for b in resp['aggregations']['by_service']['buckets']:
    print(f"  {b['key']:36s} {b['doc_count']}")
print('\nBusiest neighborhoods:')
for b in resp['aggregations']['by_neighborhood']['buckets']:
    print(f"  {b['key']:36s} {b['doc_count']}")

In [ ]:
# Median hours to close, by request type (closed cases only)
resp = es.search(index=INDEX, body={'size': 0,
    'query': {'exists': {'field': 'hours_to_close'}},
    'aggs': {'by_service': {
        'terms': {'field': 'service_name', 'size': 10},
        'aggs': {'median_hours': {'percentiles': {'field': 'hours_to_close', 'percents': [50]}}},
    }},
})
print('Median hours to close:')
for b in resp['aggregations']['by_service']['buckets']:
    print(f"  {b['key']:36s} {b['median_hours']['values']['50.0']:8.1f} h")

In [ ]:
# 311 cases within 300 m of the Elastic office (33 New Montgomery St)
resp = es.search(index=INDEX, body={'size': 0,
    'query': {'geo_distance': {'distance': '300m', 'location': {'lat': 37.7873, 'lon': -122.4005}}},
    'aggs': {'by_service': {'terms': {'field': 'service_name', 'size': 5}}},
})
print(f"Cases near the venue: {resp['hits']['total']['value']}")
for b in resp['aggregations']['by_service']['buckets']:
    print(f"  {b['key']:36s} {b['doc_count']}")

## 6. (Optional) Describe a case photo with Mistral 👀
Many cases come with a photo from the reporter. Send one to a vision-capable Mistral model to describe what's in it - a starting point for auto-triage, duplicate detection, or checking that the photo matches the request type.

> Needs your **Mistral API key**. Here we call Mistral's API directly with `requests`; see [mistral_guide.md](mistral_guide.md) for the SDK version.

In [ ]:
MISTRAL_API_KEY = '<MISTRAL_API_KEY>'  # 🔑 paste your Mistral key

def describe_photo(url, service_name):
    r = requests.post('https://api.mistral.ai/v1/chat/completions',
        headers={'Authorization': f'Bearer {MISTRAL_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'mistral-large-4', 'messages': [{'role': 'user', 'content': [
            {'type': 'text', 'text': f'This photo was attached to a San Francisco 311 "{service_name}" request. '
                                     'In one sentence, describe the problem shown.'},
            {'type': 'image_url', 'image_url': url},
        ]}]},
        timeout=60)
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

resp = es.search(index=INDEX, body={'size': 3, 'query': {'exists': {'field': 'photo_url'}},
                                            'sort': [{'requested_at': 'desc'}]})
for h in resp['hits']['hits']:
    s = h['_source']
    print(f"{s['service_name']} @ {s.get('address', '?')}\n  {s['photo_url']}")
    print(f"  → {describe_photo(s['photo_url'], s['service_name'])}\n")

## 7. Going further with Mistral

- **Semantic search:** back a `semantic_text` field with a Mistral embedding endpoint and search `status_notes` / `service_details` by meaning ("needles on the sidewalk", "tree branch about to fall"). See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).
- **Neighborhood digest:** aggregate one neighborhood's cases for the past week and have a Mistral chat model write a plain-English (or Spanish, or Chinese) summary for residents.
- **Report helper:** let someone describe a problem in their own words (or by voice with **Voxtral**), and use structured outputs to map it to the right `service_name` and find similar open cases nearby.

See [mistral_guide.md](mistral_guide.md) for calling Mistral directly.

---
## ✅ Index loaded
`sf_311_cases` is ready. Ask it "what is the Mission complaining about this month?" or "how long does graffiti removal take in my district?"

**To refresh or expand the data**, change `SAMPLE_SIZE`, `DAYS`, or the `$where` filter and re-run cells 2-4. The index is dropped and recreated each time, so there are no duplicates.